In [3]:
from pyspark.sql.functions import col, count, lower, isnan, max, when, sum
from pyspark.sql.types import *
import pytest

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 6, Finished, Available, Finished, False)

In [4]:
import logging 
logging.basicConfig(format='%(levelname)s: %(message)s', level=logging.INFO) 

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 7, Finished, Available, Finished, False)

In [5]:
def relational_check(dataframe):
    '''
    Relational checks across columns to ensure each column returns the correct value with reference to related columns.
    
    '''
    try:
        logging.info("Starting relational checks on shots, goals, power_play_goals, power_play_opportunities")

        # Goals cannot be more than shots
        invalid_goals = dataframe.filter(col("goals") > col("shots")).count()
        goals_check = invalid_goals == 0

        if goals_check:
            logging.info(f"{invalid_goals} record goals < shots, check passed")
        else:
            logging.warning(f"{invalid_goals} record goals > shots, check failed")


        # power_play_goals cannot be more than goals
        invalid_power_play_goals = dataframe.filter(col("power_play_goals") > col("goals")).count()
        power_play_goals_check = invalid_power_play_goals == 0
        
        if power_play_goals_check:
            logging.info(f"{invalid_power_play_goals} record power_play_goals < goals, check passed")
        else:
            logging.warning(f"{invalid_power_play_goals} record power_play_goals > goals, check failed")


        # If power_play_opportunities is 0, then there should not be any power_play_goals
        invalid_power_play_opportunities = dataframe.filter((col("power_play_opportunities") == 0) & (col("power_play_goals") > 0)).count()
        power_play_opportunities_check = invalid_power_play_opportunities == 0

        if power_play_opportunities_check:
            logging.info(f"{invalid_power_play_opportunities} record showing power_play_opportunities = 0 & goals > 0, check passed")
        else:
            logging.warning(f"{invalid_power_play_opportunities} record showing power_play_opportunities = 0 & goals > 0, check failed")


        result = (goals_check
                and power_play_goals_check
                and power_play_opportunities_check)

        return result

    except Exception as e:
        logging.error(f"relational_check failed: {e}")
        raise

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 8, Finished, Available, Finished, False)

In [6]:
def won_column_validation(dataframe):
    '''
    Checking combination of game_id, home_or_away, goals leads to the correct won boolean.
    
    '''
    try:
        logging.info("Checking won column validity")

        # Filter records by home and away
        df_home = dataframe.filter(col("home_or_away") == "home")
        df_away = dataframe.filter(col("home_or_away") == "away")

        # Combine home and away df on column
        df_home_away = df_home.alias("home").join(df_away.alias("away"), on="game_id", how="inner")

        # Check for home/away false win/lose
        false_win_lose_count = df_home_away.filter(((col("home.goals") > col("away.goals")) & (col("home.won") != True)) | \
                                                    ((col("home.goals") < col("away.goals")) & (col("away.won") != True)) | \
                                                    ((col("home.goals") > col("away.goals")) & (col("away.won") != False)) | \
                                                    ((col("home.goals") < col("away.goals")) & (col("home.won") != False))).count()
        false_win_lose_check = false_win_lose_count == 0

        # Check for ties
        ties = df_home_away.filter(col("home.goals") == col("away.goals")).count()
        ties_check = ties == 0

        if false_win_lose_check and ties_check:
            logging.info(f"Pass - There are {false_win_lose_count} false win/lose count and {ties} ties scenarios.")
            
        else:
            logging.warning(f"Fail - There are {false_win_lose_count} false win/lose count and {ties} ties scenarios.")

        result = (false_win_lose_check and ties_check)
        return result

    except Exception as e:
        logging.warning(f"Won column validation failed: {e}")
        raise


StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 9, Finished, Available, Finished, False)

In [7]:
def create_game_result(dataframe):
    '''
    Create new column game_result and add tie/win/lose result.
    
    '''
    try:
        logging.info("Creating new column game_result")

        # Filter records by home and away
        df_home = dataframe.filter(col("home_or_away") == "home")
        df_away = dataframe.filter(col("home_or_away") == "away")

        # Combine home and away df on column
        df_home_away = df_home.alias("home").join(df_away.alias("away"), on="game_id", how="inner")

        # create new column game_result and add tie/win/lose result for home
        df1_home = df_home_away.withColumn("game_result", when(col("home.goals") == col("away.goals"), "tie")) \
                       .withColumn("game_result", when((col("home.goals") > col("away.goals")), "win").otherwise(col("game_result"))) \
                       .withColumn("game_result", when((col("home.goals") < col("away.goals")), "lose").otherwise(col("game_result")))

        # Preparing to append game_result to original df_home dataframe
        df1_home = df1_home.select("game_id", "game_result")

        # Append game_result to df_home
        df_home = df_home.join(df1_home, on="game_id", how="left")

        # create new column game_result and add tie/win/lose result for away
        df1_away = df_home_away.withColumn("game_result", when(col("home.goals") == col("away.goals"), "tie")) \
                            .withColumn("game_result", when((col("home.goals") > col("away.goals")), "lose").otherwise(col("game_result"))) \
                            .withColumn("game_result", when((col("home.goals") < col("away.goals")), "win").otherwise(col("game_result")))

        # Preparing to append game_result to original df_away dataframe
        df1_away = df1_away.select("game_id", "game_result")

        # Append game_result to df_away
        df_away = df_away.join(df1_away, on="game_id", how="left")

        # Union rows for df_home and df_away
        df_game_result = df_home.union(df_away).sort("game_id")

        logging.info("Successfully created new column game_result")

        return df_game_result

    except Exception as e:
        logging.error(f"Creating column game_result failed: {e}")
        raise

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 10, Finished, Available, Finished, False)

In [8]:
def game_result_validation(dataframe):
    '''
    Checking combination of game_id, home_or_away, goals leads to the correct game_result value.
    
    '''
    try:
        logging.info("Checking game_result column validity")

        # Check for null
        game_result_null_count = dataframe.filter(col("game_result").isNull()).count()
        game_result_no_null = game_result_null_count == 0

        if game_result_no_null:
            logging.info(f"Pass - There are {game_result_null_count} null in game_result.")
        else: 
            logging.warning(f"Fail - There are {game_result_null_count} null in game_result.")

        # Filter records by home and away
        df_home = dataframe.filter(col("home_or_away") == "home")
        df_away = dataframe.filter(col("home_or_away") == "away")

        # Combine home and away df on column
        home_away = df_home.alias("home").join(df_away.alias("away"), on="game_id", how="inner")

        # Check for home/away false win/lose/ties
        false_win_lose_ties_count = home_away.filter(((col("home.goals") > col("away.goals")) & (col("home.game_result") != "win")) | \
                                                    ((col("home.goals") < col("away.goals")) & (col("away.game_result") != "win")) | \
                                                    ((col("home.goals") > col("away.goals")) & (col("away.game_result") != "lose")) | \
                                                    ((col("home.goals") < col("away.goals")) & (col("home.game_result") != "lose")) | \
                                                    ((col("home.goals") == col("away.goals")) & ((col("away.game_result") != "tie") | (col("home.game_result") != "tie")))).count()

        win_lose_ties_check = false_win_lose_ties_count == 0

        if win_lose_ties_check:
            logging.info(f"Pass - There are {false_win_lose_ties_count} false win/lose/ties count.")
        else:
            logging.warning(f"Fail - There are {false_win_lose_ties_count} false win/lose/ties count.")

        result = (game_result_no_null and win_lose_ties_check)
        return result

    except Exception as e:
        logging.warning(f"game_result column validation failed: {e}")
        raise

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 11, Finished, Available, Finished, False)

In [7]:
def validate_fk(child_df, child_cols, parent_df, parent_cols):
    """
    Returns child key values that do not exist
    in the parent table.
    """

    child_keys = child_df.select(*child_cols).dropDuplicates()
    parent_keys = parent_df.select(*parent_cols).dropDuplicates()

    join_condition = [
        child_keys[c] == parent_keys[p]
        for c, p in zip(child_cols, parent_cols)
    ]

    return child_keys.join(
        parent_keys,
        join_condition,
        "left_anti"
    )

StatementMeta(, 22ddb53f-3cac-42a1-a589-94c0733cbc2f, 9, Finished, Available, Finished, False)

In [8]:
# Foreign Key Validation Function
from pyspark.sql.functions import col

def fk_invalid_count(child_df, child_col, parent_df, parent_col):
    """
    Count distinct non-null child FK values
    that do not exist in the parent key.
    """
    
    child_keys = (
        child_df
        .select(child_col)
        .where(col(child_col).isNotNull())
        .dropDuplicates()
        .withColumnRenamed(child_col, "fk_value")
    )
    
    parent_keys = (
        parent_df
        .select(parent_col)
        .where(col(parent_col).isNotNull())
        .dropDuplicates()
        .withColumnRenamed(parent_col, "fk_value")
    )
    
    return child_keys.join(
        parent_keys,
        on="fk_value",
        how="left_anti"
    ).count()

StatementMeta(, fd88b6b5-1ea0-4a11-ab3e-e1a1e9a97fe8, 6, Finished, Available, Finished, False)

# **Unit test mock data**

In [9]:
test_schema = StructType([StructField("game_id", IntegerType()), 
                     StructField("team_id", IntegerType()), 
                     StructField("home_or_away", StringType()), 
                     StructField("won", BooleanType()), 
                     StructField("settled_in", StringType()), 
                     StructField("head_coach", StringType()), 
                     StructField("goals", IntegerType()), 
                     StructField("shots", IntegerType()), 
                     StructField("hits", IntegerType()), 
                     StructField("penalty_in_minutes", IntegerType()), 
                     StructField("power_play_opportunities", IntegerType()), 
                     StructField("power_play_goals", IntegerType()), 
                     StructField("face_off_win_percentage", FloatType()), 
                     StructField("giveaways", IntegerType()), 
                     StructField("takeaways", IntegerType()), 
                     StructField("blocked", IntegerType()), 
                     StructField("start_rink_side", StringType()),
                     StructField("game_result", StringType()),
                     ])

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 12, Finished, Available, Finished, False)

In [10]:
valid_mock_gts_filepath = "Files/unit_test_data/unit_test_valid_gameteamstats.csv"

valid_mock_gts = spark.read.csv(valid_mock_gts_filepath, 
                        header=True, 
                        sep=",", 
                        schema=test_schema, 
                        ignoreLeadingWhiteSpace=True, 
                        ignoreTrailingWhiteSpace=True)

valid_mock_gts.show()

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 13, Finished, Available, Finished, False)

+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|game_id|team_id|home_or_away|  won|settled_in|head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|game_result|
+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|      1|    101|        home| true|       REG|   Coach A|    4|   30|  10|                 6|                       4|               1|                   52.5|        5|        8|     12|           left|        win|
|      1|    102|        away|false|       REG|   Coach B|    2|   25|  12|                 8|                       3|             

In [11]:
invalid_mock_gts_filepath = "Files/unit_test_data/unit_test_invalid_gameteamstats.csv"

invalid_mock_gts = spark.read.csv(invalid_mock_gts_filepath, 
                        header=True, 
                        sep=",", 
                        schema=test_schema, 
                        ignoreLeadingWhiteSpace=True, 
                        ignoreTrailingWhiteSpace=True)

invalid_mock_gts.show()

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 15, Finished, Available, Finished, False)

+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|game_id|team_id|home_or_away|  won|settled_in|head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|game_result|
+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|      3|    105|        home|false|       REG|   Coach E|    2|   22|   9|                 5|                       3|               1|                   50.0|        3|        7|     11|           left|        tie|
|      3|    106|        away|false|       REG|   Coach F|    2|   24|  10|                 7|                       4|             

# **Unit test for relational_check(dataframe)**

In [12]:
# Expecting all relational checks to pass
assert relational_check(valid_mock_gts) is True

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 17, Finished, Available, Finished, False)

INFO: Starting relational checks on shots, goals, power_play_goals, power_play_opportunities
INFO: 0 record goals < shots, check passed
INFO: 0 record power_play_goals < goals, check passed
INFO: 0 record showing power_play_opportunities = 0 & goals > 0, check passed


In [13]:
# Expecting all relational checks to fail
assert relational_check(invalid_mock_gts) is False

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 18, Finished, Available, Finished, False)

INFO: Starting relational checks on shots, goals, power_play_goals, power_play_opportunities


# **Unit test for won_column_validation(dataframe)**

In [14]:
# Expecting all won value to be valid
assert won_column_validation(valid_mock_gts) is True

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 19, Finished, Available, Finished, False)

INFO: Checking won column validity
INFO: Pass - There are 0 false win/lose count and 0 ties scenarios.


In [15]:
# Expecting 3 records of won value to fail. 2 false win/lose and 1 tie.
assert won_column_validation(invalid_mock_gts) is False

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 20, Finished, Available, Finished, False)

INFO: Checking won column validity


# **Unit test for create_game_result(dataframe)**

In [16]:
# Drop game_result for valid data to test for create_game_result
invalid_mock_gts_without_gameresult = invalid_mock_gts.drop("game_result")
invalid_mock_gts_without_gameresult.show()

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 21, Finished, Available, Finished, False)

+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+
|game_id|team_id|home_or_away|  won|settled_in|head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|
+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+
|      3|    105|        home|false|       REG|   Coach E|    2|   22|   9|                 5|                       3|               1|                   50.0|        3|        7|     11|           left|
|      3|    106|        away|false|       REG|   Coach F|    2|   24|  10|                 7|                       4|               1|                   50.0|        5|        6|

In [17]:
# Expeting test to pass, game_result created
with_game_result = create_game_result(invalid_mock_gts_without_gameresult)

assert "game_result" in with_game_result.columns

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 23, Finished, Available, Finished, False)

INFO: Creating new column game_result
INFO: Successfully created new column game_result


# **Unit test for game_result_validation(dataframe)**

In [18]:
with_game_result.show()


StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 24, Finished, Available, Finished, False)

+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|game_id|team_id|home_or_away|  won|settled_in|head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|game_result|
+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|      3|    105|        home|false|       REG|   Coach E|    2|   22|   9|                 5|                       3|               1|                   50.0|        3|        7|     11|           left|        tie|
|      3|    106|        away|false|       REG|   Coach F|    2|   24|  10|                 7|                       4|             

In [19]:
# Testing game_result_validation with with_game_result dataframe
# Expecting no nulls and all value of game_result to be valid
assert game_result_validation(with_game_result) is True

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 25, Finished, Available, Finished, False)

INFO: Checking game_result column validity
INFO: Pass - There are 0 null in game_result.
INFO: Pass - There are 0 false win/lose/ties count.


In [20]:
# Preparing data for testing
# Transform invalid_mock_gts data to include null in game_result
invalid_mock_gts = invalid_mock_gts.withColumn("game_result", when(col("game_id") == 3, None).otherwise(col("game_result")))
invalid_mock_gts.show()

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 27, Finished, Available, Finished, False)

+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|game_id|team_id|home_or_away|  won|settled_in|head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|game_result|
+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|      3|    105|        home|false|       REG|   Coach E|    2|   22|   9|                 5|                       3|               1|                   50.0|        3|        7|     11|           left|       NULL|
|      3|    106|        away|false|       REG|   Coach F|    2|   24|  10|                 7|                       4|             

In [21]:
# Testing game_result_validation with invalid_mock_gts dataframe
# Expecting 1 false win/lose/tie count and 2 null
assert game_result_validation(invalid_mock_gts) is False

StatementMeta(, a033ed51-a43f-4674-a3cd-3b3a7c8ccaca, 29, Finished, Available, Finished, False)

INFO: Checking game_result column validity


# Unit test for the foreign key validation function

Testing the validation logic with small controlled Spark Dataframe.

#### Unit tests for the function

In [ ]:
def test_fk_invalid_count():

    # Parent table
    parent_df = spark.createDataFrame(
        [
            (1, "Team A"),
            (2, "Team B"),
            (3, "Team C")
        ],
        ["team_id", "team_name"]
    )

    # TEST 1: All foreign keys are valid
    child_valid = spark.createDataFrame(
        [
            (101, 1),
            (102, 2),
            (103, 3)
        ],
        ["game_id", "team_id"]
    )

    result = fk_invalid_count(
        child_valid,
        "team_id",
        parent_df,
        "team_id"
    )

    assert result == 0


    # TEST 2: One invalid FK value
    child_one_invalid = spark.createDataFrame(
        [
            (101, 1),
            (102, 2),
            (103, 99)       # invalid
        ],
        ["game_id", "team_id"]
    )

    result = fk_invalid_count(
        child_one_invalid,
        "team_id",
        parent_df,
        "team_id"
    )

    assert result == 1


    # TEST 3: Two different invalid FK values
    child_two_invalid = spark.createDataFrame(
        [
            (101, 1),
            (102, 98),      # invalid
            (103, 99)       # invalid
        ],
        ["game_id", "team_id"]
    )

    result = fk_invalid_count(
        child_two_invalid,
        "team_id",
        parent_df,
        "team_id"
    )

    assert result == 2


    # TEST 4: Repeated invalid FK
    # Should count DISTINCT invalid values only
    child_duplicate_invalid = spark.createDataFrame(
        [
            (101, 1),
            (102, 99),
            (103, 99),
            (104, 99)
        ],
        ["game_id", "team_id"]
    )

    result = fk_invalid_count(
        child_duplicate_invalid,
        "team_id",
        parent_df,
        "team_id"
    )

    assert result == 1


    print("All FK validation tests passed!")

StatementMeta(, 57be7779-e179-4bc7-9b3e-9d23b0026b1d, 13, Finished, Available, Finished, False)

In [ ]:
test_fk_invalid_count()

StatementMeta(, 57be7779-e179-4bc7-9b3e-9d23b0026b1d, 14, Finished, Available, Finished, False)

All FK validation tests passed!


The fk_invalid_count() function behaves correctly for the cases we tested, including valid keys, invalid keys, multiple invalid values, and repeated invalid values.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType

def test_fk_null_handling():

    parent_df = spark.createDataFrame(
        [(1,), (2,), (3,)],
        ["team_id"]
    )

    schema = StructType([
        StructField("game_id", IntegerType(), False),
        StructField("team_id", IntegerType(), True)
    ])

    child_df = spark.createDataFrame(
        [
            (101, 1),
            (102, 2),
            (103, None)   # NULL FK
        ],
        schema
    )

    result = fk_invalid_count(
        child_df,
        "team_id",
        parent_df,
        "team_id"
    )

    assert result == 0

    print("NULL FK handling test passed!")

StatementMeta(, fd88b6b5-1ea0-4a11-ab3e-e1a1e9a97fe8, 7, Finished, Available, Finished, False)

In [ ]:
test_fk_null_handling()

StatementMeta(, fd88b6b5-1ea0-4a11-ab3e-e1a1e9a97fe8, 8, Finished, Available, Finished, False)

NULL FK handling test passed!


The test_fk_null_handling() doesn't mean NULL is automatically acceptable. It our confirms that our function separates two different data-quality questions:  

NULL check - Is the foreign key missing?

FK validation - When a foreign key is present, does it exist in the parent table?

That's a clean design for our pipeline.